# Funciones para visualizar resultados

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict

# ──────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓN DE MÉTRICAS POR TAREA
# ──────────────────────────────────────────────────────────────────────────────

METRICAS_MOLECULAR_SUBTYPE = {
    "Clasificación": [
        "accuracy", "kappa",
        "f1_macro", "f1_micro", "f1_weighted",
        "precision_macro", "precision_micro", "precision_weighted",
        "recall_macro", "recall_micro", "recall_weighted",
    ],
    "Ranking / AUC": [
        "auroc_macro", "auroc_micro", "auroc_weighted",
        "auprc_macro", "auprc_micro", "auprc_weighted",
    ],
}

METRICAS_BINARIA = {
    "Clasificación": [
        "accuracy", "kappa",
        "f1_macro", "f1_weighted",
        "precision_macro", "precision_weighted",
        "recall_macro", "recall_weighted",
    ],
    "Ranking / AUC": [
        "auroc", "auprc",
    ],
}

TASK_LABEL = {
    "er":                "ER (binario)",
    "pr":                "PR (binario)",
    "her2":              "HER2 (binario)",
    "ki67":              "Ki67 (binario)",
    "molecular_subtype": "Subtipo molecular (4 clases)",
}

# Constantes de figura
FIG_WIDTH = 16
FIG_HEIGHT_PER_ROW = 4

# ──────────────────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────────────────

def detectar_task_desde_ruta(ruta_carpeta: str | Path) -> str | None:
    """
    Extrae la task del nombre de la carpeta buscando 'task-<nombre>'.
    Ejemplo: ...task-er_weighted... → 'er'
             ...task-molecular_subtype_weighted... → 'molecular_subtype'
    """
    nombre = os.path.basename(os.path.normpath(ruta_carpeta))
    if "task-molecular_subtype" in nombre:
        return "molecular_subtype"
    match = re.search(r'task-([a-zA-Z0-9]+?)(?:_|$)', nombre)
    if match:
        return match.group(1)
    return None


def grupo_metricas(task: str) -> dict:
    if task == "molecular_subtype":
        return METRICAS_MOLECULAR_SUBTYPE
    return METRICAS_BINARIA


def nombre_limpio(col: str) -> str:
    return col.replace("_", " ").title()


def etiqueta_corta(ruta_carpeta: str | Path) -> str:
    """
    Genera una etiqueta corta para identificar el experimento en leyendas.
    Prioridad:
      1. Nickname: texto antes del timestamp (formato YYYYMMDD_HHMMSS).
      2. Fallback: model | MIL | lr extraídos del nombre de carpeta.
    """
    nombre = os.path.basename(os.path.normpath(ruta_carpeta))

    # Si hay nickname (texto antes del timestamp), usarlo directamente
    m_nick = re.match(r'^(.+?)_\d{8}_\d{6}_', nombre)
    if m_nick:
        return m_nick.group(1)

    # Fallback: extraer las partes más informativas del nombre
    partes = []
    for patron, prefijo in [
        (r'model-([A-Za-z0-9_]+?)(?:_unfrozen|_frozen|_MIL|$)', ''),
        (r'MIL-([A-Za-z0-9]+?)(?:_|$)',                          ''),
        (r'lr([0-9e\-\.]+?)(?:_|$)',                            'lr'),
        (r'wd([0-9e\-\.]+?)(?:_|$)',                            'wd'),
    ]:
        m = re.search(patron, nombre)
        if m:
            partes.append(f"{prefijo}{m.group(1)}")
    if partes:
        return " | ".join(partes)

    return nombre[-40:] if len(nombre) > 40 else nombre


def _valor_test(exp: dict, col: str) -> float | None:
    """Devuelve el valor float de una métrica de test, o None si no existe."""
    t = exp.get("test")
    if t is not None and col in t.index:
        v = t[col]
        return float(v) if isinstance(v, (float, int, np.floating)) else None
    return None


# ──────────────────────────────────────────────────────────────────────────────
# GRÁFICAS EVOLUTIVAS TRAIN VS VAL (experimento individual)
# ──────────────────────────────────────────────────────────────────────────────

def plot_evolucion(df_train: pd.DataFrame, df_val: pd.DataFrame, task: str, ruta_carpeta: str | Path) -> None:
    grupos   = grupo_metricas(task)
    task_str = TASK_LABEL.get(task, task)

    for nombre_grupo, cols_grupo in grupos.items():
        cols = [c for c in cols_grupo if c in df_train.columns and c in df_val.columns]
        if not cols:
            continue

        n_cols = 3
        n_rows = int(np.ceil(len(cols) / n_cols))

        # squeeze=False garantiza array 2D incluso con n_rows == 1
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(FIG_WIDTH, FIG_HEIGHT_PER_ROW * n_rows), squeeze=False)
        fig.suptitle(
            f"Evolución por época — {nombre_grupo}\nTask: {task_str}",
            fontsize=14, fontweight="bold", y=1.01
        )

        axes_flat = axes.flatten()

        for i, col in enumerate(cols):
            ax = axes_flat[i]

            ax.plot(df_train["epoch"], df_train[col],
                    label="Train", color="#2563EB", linewidth=1.8,
                    marker=".", markersize=4, linestyle="-")
            ax.plot(df_val["epoch"], df_val[col],
                    label="Val", color="#DC2626", linewidth=1.8,
                    marker=".", markersize=4, linestyle="--")

            ax.set_title(nombre_limpio(col), fontsize=11)
            ax.set_xlabel("Época", fontsize=9)
            ax.set_ylabel("Valor", fontsize=9)
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.25)
            ax.set_ylim(0, 1)

            idx_best   = df_val[col].idxmax()
            best_epoch = df_val.loc[idx_best, "epoch"]
            best_val   = df_val.loc[idx_best, col]
            ax.axvline(best_epoch, color="#DC2626", linewidth=0.8,
                       linestyle=":", alpha=0.6)
            ax.annotate(f"{best_val:.3f}",
                        xy=(best_epoch, best_val),
                        xytext=(4, -12), textcoords="offset points",
                        fontsize=8, color="#DC2626")

        for j in range(len(cols), len(axes_flat)):
            axes_flat[j].axis("off")

        plt.tight_layout()
        slug     = nombre_grupo.lower().replace(" ", "_").replace("/", "")
        ruta_fig = os.path.join(ruta_carpeta, f"evolucion_{task}_{slug}.png")
        plt.savefig(ruta_fig, dpi=120, bbox_inches="tight")
        print(f"  Guardada: {ruta_fig}")
        plt.show()


# ──────────────────────────────────────────────────────────────────────────────
# TABLA DE TEST (experimento individual)
# ──────────────────────────────────────────────────────────────────────────────

def mostrar_tabla_test(ruta_carpeta: str | Path, task: str) -> None:
    ruta = os.path.join(ruta_carpeta, "test_metrics.csv")
    if not os.path.exists(ruta):
        print(f"\n  No se encontró test_metrics.csv en {ruta_carpeta}")
        return

    try:
        df     = pd.read_csv(ruta)
        fila   = df.iloc[-1]
        grupos = grupo_metricas(task)

        print(f"\n{'═'*65}")
        print(f"  RESULTADOS TEST")
        print(f"  Task: {TASK_LABEL.get(task, task)}")
        print(f"{'═'*65}")

        for nombre_grupo, cols_grupo in grupos.items():
            cols = [c for c in cols_grupo if c in fila.index]
            if not cols:
                continue

            print(f"\n  {nombre_grupo}")
            print(f"  {'Métrica':<38} │ {'Valor':>10}")
            print(f"  {'─'*38}─┼─{'─'*10}")

            for col in cols:
                valor   = fila[col]
                val_str = f"{valor:.4f}" if isinstance(valor, (float, int)) else str(valor)
                print(f"  {nombre_limpio(col):<38} │ {val_str:>10}")

        print(f"\n{'═'*65}\n")

    except Exception as e:
        print(f"  Error leyendo test_metrics.csv: {e}")


# ──────────────────────────────────────────────────────────────────────────────
# ENTRADA PRINCIPAL (experimento individual)
# ──────────────────────────────────────────────────────────────────────────────

def analizar_experimento(ruta_carpeta: str | Path) -> None:
    if not os.path.exists(ruta_carpeta):
        print(f"La ruta '{ruta_carpeta}' no existe.")
        return

    task = detectar_task_desde_ruta(ruta_carpeta)
    if task is None:
        print("No se pudo detectar la task desde el nombre de la carpeta.")
        print("Asegúrate de que el nombre contiene 'task-<nombre>'.")
        return

    print(f"\nTask detectada : {TASK_LABEL.get(task, task)}")
    print(f"Carpeta        : {os.path.basename(os.path.normpath(ruta_carpeta))}\n")

    ruta_train = os.path.join(ruta_carpeta, "train_metrics.csv")
    ruta_val   = os.path.join(ruta_carpeta, "val_metrics.csv")

    if os.path.exists(ruta_train) and os.path.exists(ruta_val):
        df_train = pd.read_csv(ruta_train).sort_values("epoch")
        df_val   = pd.read_csv(ruta_val).sort_values("epoch")
        print(f"Épocas registradas: {len(df_train)}")
        print("Generando gráficas...\n")
        plot_evolucion(df_train, df_val, task, ruta_carpeta)
    else:
        print("No se encontraron train_metrics.csv o val_metrics.csv.")

    mostrar_tabla_test(ruta_carpeta, task)


# ══════════════════════════════════════════════════════════════════════════════
# COMPARACIÓN DE MÚLTIPLES EXPERIMENTOS
# ══════════════════════════════════════════════════════════════════════════════

def _cargar_experimento(ruta_carpeta: str | Path) -> dict | None:
    """
    Carga todos los datos de un experimento y los devuelve como dict.
    Retorna None si la carpeta no es válida o no tiene task detectable.
    """
    ruta_carpeta = Path(ruta_carpeta)
    if not ruta_carpeta.exists():
        print(f"  [WARN] No existe: {ruta_carpeta}")
        return None

    task = detectar_task_desde_ruta(ruta_carpeta)
    if task is None:
        print(f"  [WARN] No se detectó task en: {ruta_carpeta}")
        return None

    datos = {
        "ruta":  str(ruta_carpeta),
        "task":  task,
        "label": etiqueta_corta(ruta_carpeta),
        "train": None,
        "val":   None,
        "test":  None,
    }

    ruta_train = ruta_carpeta / "train_metrics.csv"
    ruta_val   = ruta_carpeta / "val_metrics.csv"
    ruta_test  = ruta_carpeta / "test_metrics.csv"

    if ruta_train.exists():
        datos["train"] = pd.read_csv(ruta_train).sort_values("epoch")
    if ruta_val.exists():
        datos["val"] = pd.read_csv(ruta_val).sort_values("epoch")
    if ruta_test.exists():
        datos["test"] = pd.read_csv(ruta_test).iloc[-1]

    return datos


def _tabla_comparacion_test(experimentos_task: list[dict], task: str) -> None:
    """
    Imprime una tabla comparativa de métricas de test para un conjunto de
    experimentos de la misma task. Marca con ★ el mejor valor por métrica.
    """
    grupos = grupo_metricas(task)
    labels = [e["label"] for e in experimentos_task]

    # Recopilar las métricas disponibles en al menos un experimento
    todas_cols = []
    for nombre_grupo, cols_grupo in grupos.items():
        for col in cols_grupo:
            if any(_valor_test(e, col) is not None for e in experimentos_task):
                todas_cols.append((nombre_grupo, col))

    if not todas_cols:
        print("  No hay métricas de test disponibles.\n")
        return

    # Anchuras dinámicas
    ancho_metrica = 32
    ancho_exp     = max(14, max(len(l) for l in labels) + 2)
    ancho_total   = ancho_metrica + 3 + len(labels) * (ancho_exp + 3)

    sep_doble  = "═" * ancho_total
    sep_simple = "─" * ancho_total

    task_str = TASK_LABEL.get(task, task)
    print(f"\n{sep_doble}")
    print(f"  COMPARACIÓN TEST  │  Task: {task_str}")
    print(sep_doble)

    # Cabecera con etiquetas de experimentos
    partes = [f"  {'Métrica':<{ancho_metrica}} │"]
    partes += [f" {lbl:^{ancho_exp}} │" for lbl in labels]
    print("".join(partes))
    print(sep_simple)

    grupo_actual = None
    for nombre_grupo, col in todas_cols:

        # Separador entre grupos de métricas
        if nombre_grupo != grupo_actual:
            if grupo_actual is not None:
                print(sep_simple)
            print(f"  {f'── {nombre_grupo} ──':<{ancho_metrica}} │")
            grupo_actual = nombre_grupo

        # Valores numéricos de cada experimento
        valores  = [_valor_test(e, col) for e in experimentos_task]
        vals_validos = [v for v in valores if v is not None]
        mejor_val    = max(vals_validos) if vals_validos else None

        partes_fila = [f"  {nombre_limpio(col):<{ancho_metrica}} │"]
        for v in valores:
            if v is None:
                celda = "N/A"
            elif v == mejor_val:
                celda = f"★{v:.4f}"
            else:
                celda = f"{v:.4f}"
            partes_fila.append(f" {celda:^{ancho_exp}} │")
        print("".join(partes_fila))

    print(f"{sep_doble}")
    print("  ★ = mejor valor en esa métrica\n")


def comparar_experimentos(lista_rutas: list[str | Path]) -> None:
    """
    Compara una lista de experimentos agrupándolos por task.

    Parámetros
    ----------
    lista_rutas : list[str | Path]
        Rutas a las carpetas de los experimentos a comparar.
    Para cada task encontrada muestra:
      1. Índice de experimentos con su etiqueta y ruta completa.
      2. Tabla comparativa de métricas de test (★ = mejor valor por métrica).
    """
    # ── Cargar experimentos ──────────────────────────────────────────────────
    experimentos = []
    for ruta in lista_rutas:
        datos = _cargar_experimento(ruta)
        if datos is not None:
            experimentos.append(datos)

    if not experimentos:
        print("No se cargó ningún experimento válido.")
        return

    # ── Agrupar y ordenar por task (fuente única: TASK_LABEL) ────────────────
    por_task = defaultdict(list)
    for e in experimentos:
        por_task[e["task"]].append(e)

    orden_tasks = list(TASK_LABEL.keys())
    tasks_ordenadas = sorted(
        por_task.keys(),
        key=lambda t: orden_tasks.index(t) if t in orden_tasks else 99
    )

    print(f"\n{'='*70}")
    print(f"  COMPARACIÓN DE {len(experimentos)} EXPERIMENTOS")
    n_tasks = len(tasks_ordenadas)
    print(f"  Tasks ({n_tasks}): {', '.join(TASK_LABEL.get(t, t) for t in tasks_ordenadas)}")
    print(f"{'='*70}")

    for task in tasks_ordenadas:
        exps     = por_task[task]
        task_str = TASK_LABEL.get(task, task)
        n        = len(exps)

        print(f"\n\n{'━'*70}")
        print(f"  TASK: {task_str}  ({n} experimento{'s' if n > 1 else ''})")
        print(f"{'━'*70}")
        for k, e in enumerate(exps):
            print(f"  [{k}] {e['label']}")
            print(f"       {e['ruta']}")
        print()

        # 1. Tabla comparativa de test
        _tabla_comparacion_test(exps, task)


# ──────────────────────────────────────────────────────────────────────────────
# EJECUCIÓN
# ──────────────────────────────────────────────────────────────────────────────

# # ── Analizar un experimento individual ───────────────────────────────────────
# ruta = r"/workspace/results/DINOV1_2_224_CLS_20260328_124302_task-er_weighted_loss_model-VIT_frozen_MIL-attentionMIL_AdamW_lr1e-05_wd0.001_bs32_ep100"
# analizar_experimento(ruta)

# # ── Analizar todos los experimentos de una carpeta (individualmente) ─────────
# for ruta in Path("/workspace/results").iterdir():
#     if ruta.is_dir():
#         analizar_experimento(ruta)

# ── Comparar una lista concreta de experimentos ───────────────────────────────
comparar_experimentos([
    "/workspace/results/DINOV1_2_224_CLS_20260328_124302_task-er_weighted_loss_model-VIT_frozen_MIL-attentionMIL_AdamW_lr1e-05_wd0.001_bs32_ep100",
    "/workspace/results/DINOV1_2_224_MEANPOOLING_20260328_124242_task-er_weighted_loss_model-VIT_frozen_MIL-attentionMIL_AdamW_lr1e-05_wd0.001_bs32_ep100",
    "/workspace/results/DINOV1_224_CLS_20260324_160304_task-ki67_weighted_loss_model-VIT_frozen_MIL-attentionMIL_AdamW_lr1e-05_wd0.001_bs32_ep100",
    "/workspace/results/DINOV2_224_MEANPOOLING_20260328_020820_task-ki67_weighted_loss_model-VIT_frozen_MIL-attentionMIL_AdamW_lr1e-05_wd0.001_bs32_ep100",
    "/workspace/results/DINOV2_224_MEANPOOLING_20260328_071813_task-molecular_subtype_weighted_loss_model-VIT_frozen_MIL-attentionMIL_AdamW_lr1e-05_wd0.001_bs32_ep100"
])
